In [ ]:
import pandas as pd
from datetime import timedelta
from config import CSV_FILE_APP, CSV_FILE_PHY, TIME_INFO_FILE, KPI_OUTPUT_FOLDER


# === FILE PATHS ===
# app_kpis_path = 'APP_KPIs/single_tone_gain_30_fft_1024.csv'
# phy_kpis_path = 'PHY_KPIs/single_tone_gain_30_fft_1024.csv'
# time_info_path = 'spectrograms/single_tone_gain_30_fft_1024/time_info.csv'

app_kpis_path = CSV_FILE_APP
phy_kpis_path = CSV_FILE_PHY
time_info_path = TIME_INFO_FILE

# === NORMALIZATION (MIN/MAX) CONSTANTS ===
# For APP_KPIs
MIN_LATENCY = 0
MAX_LATENCY = 60 * 10  # 600

MIN_JITTER = 0
MAX_JITTER = 5 * 10    # 50

MIN_LOST_PACKETS = 0
MAX_LOST_PACKETS = 10

# For PHY_KPIs
MIN_SNR = 45
MAX_SNR = 65

MIN_NOISE = -105
MAX_NOISE = -95


# === LOAD FILES ===
app_df = pd.read_csv(app_kpis_path)
phy_df = pd.read_csv(phy_kpis_path)
time_info_df = pd.read_csv(time_info_path)

# === PARSE 'Time' COLUMNS ===
def parse_time_column(df):
    return pd.to_datetime(df['Time'], format='%H:%M:%S.%f')

app_df['Time'] = parse_time_column(app_df)
phy_df['Time'] = parse_time_column(phy_df)
time_info_df['Time'] = parse_time_column(time_info_df)

# === GET TIME RANGE FROM SPECTROGRAMS ===
first_time = time_info_df['Time'].iloc[0]
last_time = time_info_df['Time'].iloc[-1]

# === EXPAND WINDOW BY 20 SECONDS ===
window_before = timedelta(seconds=20)
adjusted_start_time = first_time - window_before

print(f"Filtering from {adjusted_start_time.time()} to {last_time.time()}")

# === FILTER APP_KPIs and PHY_KPIs BASED ON TIME ===
filtered_app_df = app_df[(app_df['Time'] >= adjusted_start_time) & (app_df['Time'] <= last_time)]
filtered_phy_df = phy_df[(phy_df['Time'] >= adjusted_start_time) & (phy_df['Time'] <= last_time)]

# === FILTER OUT OUTLIERS IN PHY_KPIs ===
filtered_phy_df = filtered_phy_df[
    (filtered_phy_df['Noise'] >= MIN_NOISE) & (filtered_phy_df['Noise'] <= MAX_NOISE) &
    (filtered_phy_df['SNR'] >= MIN_SNR) & (filtered_phy_df['SNR'] <= MAX_SNR)
]


In [ ]:
# === NORMALIZE FUNCTION ===
def min_max_normalize(series, min_val, max_val):
    normalized = (series - min_val) / (max_val - min_val)
    return normalized.clip(0, 1)

# === NORMALIZE APP KPIs ===
filtered_app_df['Latency'] = min_max_normalize(filtered_app_df['Latency'], MIN_LATENCY, MAX_LATENCY)
filtered_app_df['Jitter'] = min_max_normalize(filtered_app_df['Jitter'], MIN_JITTER, MAX_JITTER)
filtered_app_df['Packet Loss Count'] = min_max_normalize(filtered_app_df['Packet Loss Count'], MIN_LOST_PACKETS, MAX_LOST_PACKETS)

# === NORMALIZE PHY KPIs ===
filtered_phy_df['SNR'] = min_max_normalize(filtered_phy_df['SNR'], MIN_SNR, MAX_SNR)
filtered_phy_df['Noise'] = min_max_normalize(filtered_phy_df['Noise'], MIN_NOISE, MAX_NOISE)

# === FORMAT 'Time' BACK TO HH:MM:SS.mmm ===
filtered_app_df['Time'] = filtered_app_df['Time'].dt.strftime('%H:%M:%S.%f').str[:-3]
filtered_phy_df['Time'] = filtered_phy_df['Time'].dt.strftime('%H:%M:%S.%f').str[:-3]

# === SAVE BACK TO CSV ===
filtered_app_df.to_csv(app_kpis_path, index=False)
filtered_phy_df.to_csv(phy_kpis_path, index=False)

print(f"Filtered and normalized APP_KPIs: {len(filtered_app_df)} rows kept")
print(f"Filtered and normalized PHY_KPIs: {len(filtered_phy_df)} rows kept")